# IoU-F1 Evaluation
Convert chunk predictions to laugh segments, compute IoU-based F1 vs EMNLP ground truth
Compare against StandUp4AI: IoU-F1=0.51 @ IoU=0.2

In [ ]:
# Setup
from google.colab import drive
drive.mount('/content/drive')

import os, json, numpy as np, pandas as pd
from sklearn.metrics import f1_score, precision_score, recall_score

BASE = '/content/drive/MyDrive/standup4ai'
FEAT_DIR = BASE + '/features_221'
LABEL_DIR = BASE + '/seq-Standup4AI/dataset/en_uk/emnlp+jahak/train'
MODEL_PATH = BASE + '/wordlevel_221_model.pt'

print('Setup complete')


In [ ]:
# Reload model + data (same as training)
import torch, torch.nn as nn
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupKFold

class WordModel(nn.Module):
    def __init__(self, in_dim=791):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 256), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(256, 64), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(64, 16), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(16, 1), nn.Sigmoid()
        )
    def forward(self, x):
        return self.net(x)

def parse_timestamp(ts_str):
    ts_str = str(ts_str).strip()
    try:
        parts = ts_str.strip('[]').split(',')
        return float(parts[0]), float(parts[1])
    except:
        return None, None

# Load features + labels (timestamp-based mapping)
feat_files = sorted([f for f in os.listdir(FEAT_DIR) if f.endswith('_features.npy')])
X_list, y_list, vids, all_labels = [], [], [], []

for f in feat_files:
    vid = f.replace('_features.npy', '')
    label_path = LABEL_DIR + '/' + vid + '.csv'
    if not os.path.exists(label_path):
        continue
    feats = np.load(FEAT_DIR + '/' + f)
    labels_df = pd.read_csv(label_path)
    n_chunks = len(feats)
    chunk_dur = 5.0

    word_times, word_labels = [], []
    for _, row in labels_df.iterrows():
        t0, t1 = parse_timestamp(row['timestamp'])
        if t0 is not None:
            word_times.append((t0, t1))
            word_labels.append(str(row['label']).strip())

    chunk_labels = []
    for i in range(n_chunks):
        c0, c1 = i * chunk_dur, (i+1) * chunk_dur
        is_laugh = any(
            wl in ['B', 'I', 'L'] and w0 < c1 and w1 > c0
            for (w0, w1), wl in zip(word_times, word_labels)
        )
        chunk_labels.append(1 if is_laugh else 0)
    
    X_list.append(feats)
    y_list.append(np.array(chunk_labels))
    vids.extend([vid] * n_chunks)
    all_labels.append(labels_df)

X = np.vstack(X_list)
y = np.concatenate(y_list)
groups = np.array(vids)

print('X:', X.shape, 'y:', y.shape, 'pos rate:', round(y.mean(), 3))
print('Unique videos:', len(set(vids)))


In [ ]:
# Load trained model
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = WordModel().to(device)
if os.path.exists(MODEL_PATH):
    model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
    print('Model loaded')
else:
    print('WARNING: Model not found at', MODEL_PATH)
    print('Using XGBoost instead...')
    import xgboost as xgb
    from sklearn.preprocessing import StandardScaler
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    model = xgb.XGBClassifier(n_estimators=200, max_depth=4, learning_rate=0.05,
                              use_label_encoder=False, eval_metric='logloss', verbosity=0)
    model.fit(X_scaled, y)
    probs_all = model.predict_proba(X_scaled)[:, 1]
    print('XGBoost trained')
else:
    model.eval()
    with torch.no_grad():
        probs_all = model(torch.tensor(X, dtype=torch.float32).to(device)).cpu().numpy().squeeze()

print('Predictions done')


In [ ]:
# IoU evaluation functions
def get_gt_segments(labels_df):
    """Extract ground-truth laugh segments from EMNLP BIO labels"""
    segments = []
    i = 0
    while i < len(labels_df):
        lbl = str(labels_df.iloc[i]['label']).strip()
        t0, t1 = parse_timestamp(labels_df.iloc[i]['timestamp'])
        if t0 is None:
            i += 1
            continue
        if lbl == 'L':  # Single-word laugh
            segments.append((t0, t1))
        elif lbl == 'B':  # Multi-word laugh start
            start, end = t0, t1
            j = i + 1
            while j < len(labels_df):
                nl = str(labels_df.iloc[j]['label']).strip()
                if nl in ('I', 'L'):
                    _, end = parse_timestamp(labels_df.iloc[j]['timestamp'])
                    j += 1
                else:
                    break
            segments.append((start, end))
            i = j - 1
        i += 1
    return segments

def span_iou(s1, s2):
    """IoU of two temporal spans"""
    inter = max(0, min(s1[1], s2[1]) - max(s1[0], s2[0]))
    union = max(s1[1], s2[1]) - min(s1[0], s2[0])
    return inter / union if union > 0 else 0

def chunk_preds_to_segments(probs, timestamps, threshold=0.5):
    """Merge consecutive above-threshold chunks into laugh segments"""
    segments = []
    in_seg = False
    seg_start = 0
    for i, (p, (t0, t1)) in enumerate(zip(probs, timestamps)):
        if p >= threshold and not in_seg:
            in_seg = True
            seg_start = t0
        elif p < threshold and in_seg:
            in_seg = False
            segments.append((seg_start, t0))
    if in_seg:
        segments.append((seg_start, timestamps[-1][1]))
    return segments

def segment_f1(pred_segs, gt_segs, iou_thresh=0.2):
    """Compute segment-level P/R/F1 at given IoU threshold"""
    if not pred_segs:
        return 0.0, 0.0, 0.0
    if not gt_segs:
        return 0.0, 0.0, 0.0
    matched_pred, matched_gt = set(), set()
    for pi, ps in enumerate(pred_segs):
        best_iou, best_gi = 0.0, -1
        for gi, gs in enumerate(gt_segs):
            if gi in matched_gt:
                continue
            iou_val = span_iou(ps, gs)
            if iou_val >= iou_thresh and iou_val > best_iou:
                best_iou, best_gi = iou_val, gi
        if best_gi >= 0:
            matched_pred.add(pi)
            matched_gt.add(best_gi)
    tp = len(matched_pred)
    p = tp / len(pred_segs) if pred_segs else 0.0
    r = tp / len(gt_segs) if gt_segs else 0.0
    f = 2 * p * r / (p + r) if (p + r) > 0 else 0.0
    return p, r, f

print('IoU functions defined')


In [ ]:
# Evaluate per video at multiple IoU thresholds
IOU_THRESHOLDS = [0.1, 0.2, 0.3, 0.4, 0.5]

# Get predictions per video
feat_files = sorted([f for f in os.listdir(FEAT_DIR) if f.endswith('_features.npy')])

if 'model' in dir() and hasattr(model, 'predict_proba'):
    probs_all = model.predict_proba(X_scaled)[:, 1]
else:
    with torch.no_grad():
        probs_all = model(torch.tensor(X, dtype=torch.float32).to(device)).cpu().numpy().squeeze()

idx = 0
results_by_iou = {th: [] for th in IOU_THRESHOLDS}
per_video = []

for f in feat_files:
    vid = f.replace('_features.npy', '')
    label_path = LABEL_DIR + '/' + vid + '.csv'
    if not os.path.exists(label_path):
        continue
    
    feats = np.load(FEAT_DIR + '/' + f)
    labels_df = pd.read_csv(label_path)
    n_chunks = len(feats)
    chunk_dur = 5.0
    
    # Get predictions for this video
    probs = probs_all[idx:idx+n_chunks]
    idx += n_chunks
    
    # Build timestamps
    timestamps = [(i * chunk_dur, (i+1) * chunk_dur) for i in range(n_chunks)]
    
    # Predicted segments
    pred_segs = chunk_preds_to_segments(probs, timestamps, threshold=0.5)
    
    # Ground truth segments
    gt_segs = get_gt_segments(labels_df)
    
    row = {'vid': vid, 'n_pred': len(pred_segs), 'n_gt': len(gt_segs)}
    for th in IOU_THRESHOLDS:
        p, r, f = segment_f1(pred_segs, gt_segs, th)
        row[f'p_{th}'] = round(p, 4)
        row[f'r_{th}'] = round(r, 4)
        row[f'f_{th}'] = round(f, 4)
        results_by_iou[th].append({'vid': vid, 'p': p, 'r': r, 'f': f})
    per_video.append(row)

print('Evaluated', len(per_video), 'videos')


In [ ]:
# Results summary
print('=' * 60)
print('IoU-F1 EVALUATION RESULTS')
print('=' * 60)
print(f"{'IoU':>6} | {'Precision':>10} {'Recall':>10} {'F1':>10}")
print('-' * 45)

summary = {}
for th in IOU_THRESHOLDS:
    rs = results_by_iou[th]
    if not rs:
        continue
    pm = np.mean([x['p'] for x in rs])
    rm = np.mean([x['r'] for x in rs])
    fm = np.mean([x['f'] for x in rs])
    summary[th] = {'p': pm, 'r': rm, 'f': fm}
    print(f" >= {th:.1f} | {pm:.4f} {rm:.4f} {fm:.4f}")

print('')
print('Comparison: StandUp4AI (EMNLP 2025): IoU-F1=0.51 @ IoU=0.2')
print('')

# Best IoU threshold
best_iou = max(summary.keys(), key=lambda th: summary[th]['f'])
print(f'Best IoU threshold: {best_iou:.1f} with F1={summary[best_iou]["f"]:.4f}')

# Per-video sample
print('')
print('Top 10 videos by IoU-F1@0.2:')
print(f"{'Video':<20} {'n_pred':>6} {'n_gt':>5} {'P':>8} {'R':>8} {'F1':>8}")
print('-' * 60)
for row in sorted(per_video, key=lambda x: x.get('f_0.2', 0), reverse=True)[:10]:
    print(f"{row['vid']:<20} {row['n_pred']:>6} {row['n_gt']:>5} "
          f"{row.get('p_0.2', 0):>8.4f} {row.get('r_0.2', 0):>8.4f} {row.get('f_0.2', 0):>8.4f}")

# Save results
import json
out = {
    'iou_thresholds': IOU_THRESHOLDS,
    'summary': summary,
    'per_video': per_video
}
with open(BASE + '/iou_f1_results.json', 'w') as f:
    json.dump(out, f, indent=2)
print('')
print('Results saved to', BASE + '/iou_f1_results.json')
